In [2]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. The Skill Template
class Skill(BaseModel):
    name: str = Field(description="Name of the skill (e.g. Python, SQL, Project Management)")
    context: str = Field(description="Briefly how/where it was used")
    years_of_experience: Optional[int] = Field(description="Years used, if mentioned")
    proficiency: str = Field(description="Level: Beginner, Intermediate, or Expert")

# 2. The Master Template
class ExtractedData(BaseModel):

    job_title: str = Field(description="The formal job title found in the text")
    education: List[str] = Field(description="Degrees, majors, and universities attended")
    tools_and_platforms: List[str] = Field(description="Software, cloud providers, or tools (e.g. AWS, Jira, Tableau)")
    hard_skills: List[Skill] = Field(description="Technical skills and programming languages like python, java, sql, etc.")
    soft_skills: List[Skill] = Field(description="Interpersonal and leadership skills")
    certifications: List[str] = Field(description="Any professional certifications or licenses they have got certified in")
    total_years_experience: float = Field(description="Total years of work experience mentioned")

# 3. Setup the Agent 
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

# Binding the schema to the LLM
parser_agent = llm.with_structured_output(ExtractedData)

## Agent 1

In [3]:
dummy_resume = """
Mehar Singh Bawa
bawamehar@gmail.com +1 9705661937 linkedin.com/in/bawamehar/ github.com/bawamehar
PROFILE
Data Specialist with a proven track record of optimizing business processes and reducing operational risk. Brings
5 years of industry experience in diagnostics and process automation, now applied to Predictive Modeling and
Data Visualization. Expert in bridging the gap between technical metrics and business goals, seeking to leverage
advanced analytics skills to solve complex data challenges.
EDUCATION
Masters in Computer Information Systems
Colorado State University
01/2025 – Present
•Coursework: Business Intelligence, Applied Data Mining and Analytics, Enterprise Resource Planning.
•Graduate Assistant for Analytics and AI in Business.
•Member of Dean’s List for Spring 2025, Fall 2026.
B-Tech in Computer Science and Engineering
SRM Institute of Science and Technology
05/2019
MACHINE LEARNING & DATA PROJECTS
Airline Passenger Satisfaction 12/2025
•Predicted airline passenger satisfaction using machine learning.Includes data preprocessing, visualization,
feature scaling, multiple models (Logistic Regression,Random Forest,Adaboost, etc), cross-validation, and
feature importance analysis.
Diabetes Prediction Project
•Architected a ML application featuring a Streamlit UI and FastAPI backend to automate real-time diabetes risk
assessment and store patient data in PostgreSQL.
•Developed a training pipeline that evaluates multiple classifiers models, exports the best model via Joblib, and
uses MLOps for production orchestration and model versioning.
SAP ERP Simulation Game, Colorado State University
CIS601 Enterprise Computing & SystemsIntegration
05/2025
•Managed a Muesli Manufacturing Company using SAP ERP, engaging in complex business processes from raw
material acquisition to product sales. Leveraged Microsoft Power BI to analyze and interpret data, making datadriven decisions that led to a competitive advantage over rivals and resulted in winning the game.
Electric Vehicle Analysis, Colorado State University
CIS570 BusinessIntelligence
05/2025
•Conducted data analysis on EV adoption, sales trends, charging efficiency, economic benefits and impact using
Power BI, creating interactive dashboards to highlight patterns and support data-driven decision-making.
PROFESSIONAL EXPERIENCE
Consultant
Capgemini
02/2020 – 01/2025 | Pune,IN
•Developed multiple automated processes to minimize the manual tasks and enhanced the service delivery.
•Maintained high-availability production environments on AWS, utilizing Dynatrace for root-cause analysis of
system outages.
•Translated vague client requirements into measurable KPIs, using data analysis to prioritize features and
optimize delivery.
•Analysed and visualized historical failure patterns using FMEA methodology, interpreting key data trends to
optimize business processes and achieve a 70% reduction in problems.
SKILLS & CERTIFICATIONS
Skills: Python (pandas, scikit-learn), AWS, Power BI, Snowflake, R, Tableau, Knime, SQL, Dynatrace, LangChain
Certification: AWS Certified Developer- Associate, Certified SAFe 6 Scrum Master
"""

dummy_jd = """
Job Description

**This is a fully on-site role. Hybrid/remote work is not available at this time**

**We are unable to sponsor or take over sponsorship of an employment Visa at this time**




Are you early in your data science career and eager to work on real, production‑impacting models? RIVO is excited to find our next Junior Data Scientist to join our Data Team and help build the data pipelines, features, and model experiments that power our decision‑making tools within our Risk & Underwriting teams.


This is a great role for someone who is eager to learn and be mentored by senior team members, loves hands‑on data work, and has a knack for solving analytical problems. If you're excited about solving meaningful problems with data — and growing your skills along the way — we’d love to hear from you!




What You’ll Do:


Build and maintain feature pipelines that support risk and underwriting models.
Use Python and SQL to explore, transform, and prepare datasets for modeling.
Assist with training, validating, and monitoring predictive models — including probability‑of‑default and other underwriting scores.
Support A/B testing and champion/challenger experiments to evaluate new features and model improvements.
Look for opportunities to apply AI tools to streamline workflows and enhance analysis
Identify and troubleshoot data quality issues such as anomalies, schema drift, or inconsistent signals.
Collaborate with Underwriting, Data Engineering, and Analytics partners to translate business questions into analytical solutions.
Contribute to improving our model infrastructure, pipelines, and analytical workflows.
Why You’ll Love Working Here:


Collaborative team culture with strong mentorship to grow your data science skills
Work directly on high‑impact models used across the business
Exposure to real‑world ML workflows in production
Opportunities to grow into more advanced modeling, experimentation, or analytics engineering paths

Qualifications

What You Bring:


Bachelor’s degree in Data Science, Computer Science, Statistics, Mathematics, Economics, Engineering, or a related quantitative field required.
1–3 years of experience (including internships) in data science, analytics, or a similar applied role required.
A strong foundation in Python and SQL for data manipulation and modeling.
Solid understanding of core machine learning concepts (classification, validation methods, evaluation metrics).
Familiarity with EDA and hands‑on experience working with structured (and ideally some unstructured) datasets.
A growth mindset — you’re eager to learn, ask thoughtful questions, and proactively seek clarity when needed.
A natural sense of curiosity and genuine passion for data science and machine learning.
Willingness to upskill independently, explore new tools or techniques, and bring fresh ideas to the team.
A proactive interest in using AI tools to learn faster, work smarter, and boost team impact.
Strong attention to detail and an appreciation for clean, well‑structured, reliable data.
Comfortable communicating your findings clearly to your manager and project partners.
Experience with Git/GitHub is a plus (or excitement to learn it quickly).

"""

resume_input = {"doc_type": "Resume", "raw_text": dummy_resume}
jd_input = {"doc_type": "Job Description", "raw_text": dummy_jd}

In [4]:
system_prompt = """
You are an expert Technical Recruiter with 15 years of experience in talent acquisition. 
Your task is to analyze the provided text (either a Resume or a Job Description) and extract 
structured information with high precision.

### INSTRUCTIONS:
1. **Contextual Extraction**: Don't just list keywords. Understand how a skill was used.
2. **Date Math**: If dates are provided (e.g., Jan 2020 - Dec 2023), calculate the duration in years. 
   Total all relevant periods to provide the 'total_years_experience'.
3. **Skill Classification**: 
   - Hard Skills: Programming languages, frameworks, and core methodologies.
   - Tools/Platforms: Specific software like AWS, Docker, Jira, or Snowflake.
4. **No Hallucinations**: Only extract what is explicitly stated or strongly implied by the context. 
   If a piece of info (like a certification) is missing, return an empty list.

**Specific Extraction Rules**:
   - CERTIFICATIONS: Lprobably ook at the end of the document. 
   - EXPERIENCE: Calculate years based on the intro oparagraph mostly on the starting of the document or calculate based on dates (2020-2025 = 5 years). 
   - SKILLS: See Skills section properly and extract all the skills for their.

Extract the information into the required JSON format.
"""

# In your code, you'd pass a variable 'doc_type' which is either "Resume" or "Job Description"
user_prompt = """
I have provided a {doc_type} below. 
Please analyze it and extract the data into the JSON format specified in the schema. 

Ensure that if this is a Job Description, you extract the 'requirements', 
and if it is a Resume, you extract the 'attainments'.

{doc_type} Text:
{raw_text}
"""

In [ ]:
# parser_agent = llm.with_structured_output(ExtractedData)
# test_text = "Experienced Data Scientist with 5 years in Python and a Masters in Stats from CSU."
# result = parser_agent.invoke(test_text)

f:\langchain_proj\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ExtractedData(job_title='...al_years_experience=5.0), input_type=ExtractedData])
  return self.__pydantic_serializer__.to_python(


In [15]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the specific "Mission Control" for Agent 1
parser_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt), 
    ("user", user_prompt)
])

# 2. Chain them together
# This is the "LangChain" way: Prompt -> Model -> Structured Output
parser_chain = parser_prompt | parser_agent 


#test_resume = "Experienced Data Scientist with 5 years in Python and a Masters in Stats from CSU."


resume_json = parser_chain.invoke(resume_input)
jd_json = parser_chain.invoke(jd_input)

print("--- RESUME DATA ---")
print(resume_json.model_dump_json(indent=2))

--- RESUME DATA ---
{
  "job_title": "Data Specialist",
  "education": [
    "Masters in Computer Information Systems, Colorado State University (01/2025 – Present)",
    "B-Tech in Computer Science and Engineering, SRM Institute of Science and Technology (05/2019)"
  ],
  "tools_and_platforms": [
    "AWS",
    "Power BI",
    "Snowflake",
    "Dynatrace",
    "LangChain",
    "Streamlit",
    "FastAPI",
    "PostgreSQL",
    "Joblib",
    "MLOps",
    "SAP ERP",
    "Tableau",
    "Knime"
  ],
  "hard_skills": [
    {
      "name": "Python (pandas, scikit-learn)",
      "context": "Used for data preprocessing, feature scaling, model training and evaluation across predictive modeling and ML projects (e.g., airline satisfaction, diabetes prediction).",
      "years_of_experience": null,
      "proficiency": "Expert"
    },
    {
      "name": "Machine Learning",
      "context": "Built and evaluated multiple classifiers (Logistic Regression, Random Forest, AdaBoost, etc.), performed cr

f:\langchain_proj\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ExtractedData(job_title='...al_years_experience=1.0), input_type=ExtractedData])
  return self.__pydantic_serializer__.to_python(


In [16]:
print("\n--- JD DATA ---")
print(jd_json.model_dump_json(indent=2))


--- JD DATA ---
{
  "job_title": "Junior Data Scientist",
  "education": [
    "Bachelor’s degree in Data Science",
    "Bachelor’s degree in Computer Science",
    "Bachelor’s degree in Statistics",
    "Bachelor’s degree in Mathematics",
    "Bachelor’s degree in Economics",
    "Bachelor’s degree in Engineering",
    "Bachelor’s degree in a related quantitative field"
  ],
  "tools_and_platforms": [
    "Git/GitHub",
    "AI tools (general)"
  ],
  "hard_skills": [
    {
      "name": "Python",
      "context": "Use Python to explore, transform, and prepare datasets for modeling; build and maintain feature pipelines.",
      "years_of_experience": null,
      "proficiency": "Intermediate"
    },
    {
      "name": "SQL",
      "context": "Use SQL to explore, transform, and prepare datasets for modeling; support feature pipelines for risk and underwriting models.",
      "years_of_experience": null,
      "proficiency": "Intermediate"
    },
    {
      "name": "Feature engineering

## Agent 2

In [17]:
class MissingSkill(BaseModel):
    name: str = Field(description="Skill name found in JD but not in Resume")
    is_dealbreaker: bool = Field(description="True if listed in requirements or 'must haves', or mentioned multiple times, False if a 'plus', or 'good to have, etc'")
    priority: int = Field(description="1 for highest priority, 3 for lowest")

class MatchResults(BaseModel):
    match_score: int = Field(description="0-100 technical alignment score")
    experience_reasoning: str = Field(description="Evaluation of YOE using the 20/50 rule or Detailed logic on YOE match")
    skill_reasoning: str = Field(description="Logic on Must-Have vs Nice-to-Have alignment")
    project_reasoning: str = Field(description="How well projects/tools prove the required competencies and match the JD needs")
    missing_skills: List[MissingSkill] = Field(description="List of missing technical/soft skills") # Priority-sorted list
    score_justification: str = Field(description="Comprehensive breakdown of how the final score was calculated")
    education_reasoning: str = Field(description="How the candidate's degree level and certifications match the JD requirements")
    

In [18]:
matcher_system_prompt = """
You are an expert Senior Technical Recruiter and Matchmaking Agent. Your mission is to perform a deep-dive comparison between a candidate's profile (resume_json) and a job's requirements (jd_json).

### SCORING SYSTEM:
Your baseline is 100 points. Apply the following deductions and logic:

1. **Years of Experience (YOE) - The 20/50 Rule**:
   - Within 20% of required YOE: No deduction.
   - 20% to 50% less than required YOE: Deduct 15 points.
   - More than 50% less than required YOE: Deduct 30 points.
   - If candidate exceeds YOE: Treat as a Strength (0 deduction).

2. **Skill Alignment**:
   - Missing a 'Must-Have/Required' skill: Deduct 10 points per skill.
   - Missing a 'Nice-to-Have/Plus' skill: Deduct 2 points per skill.
   - Note: Use contextual matching. If a candidate uses a tool for a similar purpose (e.g., Tableau vs Power BI), acknowledge the overlap but note the specific tool gap.

3. **Education & Certifications**:
   - Evaluate the candidate's highest level of education against the JD's requirements.
   - Required Degree not met: Deduct 20 points.
   - Higher Degree than required: Treat as a Strength (+5 virtual points to offset other gaps).
   - Relevant Certifications: Use these to validate skill proficiency even if professional years are low.

### OUTPUT INSTRUCTIONS:
- Analyze every field in the provided JSON objects.
- Categorize 'missing_skills' by their 'is_dealbreaker' status. Also, Ensure 'missing_skills' is a sorted list where Priority 1 (Dealbreakers) appears at the top.
- Provide separate, detailed reasoning for Experience, Skills, Projects, and Education.
- Ensure 'score_justification' explains the math (e.g., "Score of 85: -15 for YOE gap, but +5 for Master's degree").

Maintain a professional, objective tone. Do not hallucinate data.
"""

In [20]:
# Create the Matcher Agent
matcher_agent = llm.with_structured_output(MatchResults)

matcher_user_prompt = """
RESUME DATA:
{resume_json}

JOB DESCRIPTION DATA:
{jd_json}

Provide a detailed Match score.
"""

# Prepare the input using your parsed JSON data
user_content = matcher_user_prompt.format(
    resume_json=resume_json.model_dump_json(),
    jd_json=jd_json.model_dump_json()
)

# Call the model
match_output = matcher_agent.invoke([
    ("system", matcher_system_prompt),
    ("user", user_content)
])

f:\langchain_proj\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MatchResults(match_score=...and process knowledge."), input_type=MatchResults])
  return self.__pydantic_serializer__.to_python(


In [21]:
def print_basic_report(results):
    print(f"--- MATCH SCORE: {results.match_score}/100 ---")
    print(f"\n[EXP LOGIC]: {results.experience_reasoning}")
    print(f"[SKILL LOGIC]: {results.skill_reasoning}")
    print(f"[PROJECT LOGIC]: {results.project_reasoning}")
    print(f"[EDUCATION LOGIC]: {results.education_reasoning}")
    
    print("\n--- MISSING SKILLS (Sorted) ---")
    for s in results.missing_skills:
        status = "DEALBREAKER" if s.is_dealbreaker else "Nice-to-have"
        print(f"- {s.name} (Priority {s.priority}): {status}")
    
    print(f"\n--- FINAL AUDIT ---\n{results.score_justification}")

# Run the display
print_basic_report(match_output)

--- MATCH SCORE: 75/100 ---

[EXP LOGIC]: JD requests ~1 year (Junior). Candidate has 5 years of professional experience and is currently pursuing a Master's. Per the 20/50 rule this exceeds the requirement — treated as a strength (no deduction). No YOE penalty applied.
[SKILL LOGIC]: Strong alignment on core technical skills: Python (expert), SQL/PostgreSQL (intermediate), feature engineering, model training/validation, core ML concepts, EDA, MLOps/model deployment, and data preprocessing. These map directly to most JD requirements. Missing or not evidenced: (1) Domain modeling for underwriting / probability-of-default — no experience or project shown in resume specific to underwriting/PD models; (2) A/B testing and experimentation — no mention of running or supporting A/B tests or champion/challenger experiments; (3) Git/GitHub — not listed in the resume tools/platforms. Each missing item above is treated as a required JD skill and assessed as a dealbreaker for scoring (-10 points ea

In [ ]:
from typing import TypedDict, Optional

class AgentState(TypedDict):
    # Inputs
    raw_resume: str
    raw_jd: str
    
    # Agent 1 Output (The Foundation)
    resume_data: ExtractedData 
    jd_data: ExtractedData
    
    # Agent 2 Output (The Matchmaker)
    match_results: MatchResults
    
    # Agent 3 Output (The ATS Specialist)
    ats_results: dict # We'll define this later
    
    # Agent 4 Output (The Final Report)
    final_feedback: str

## Agent 3

state  
push  